# Replay Extracted Code Blocks on a New Clock

This notebook demonstrates how to **replay** agent-generated code blocks on a different aging clock column — running the same analysis pipeline without any LLM calls.

**Key idea:** The extracted code blocks have hardcoded clock column names (e.g. `Hazard_Rodents_All_All_scaleddiff_EN_tAge`). To replay on a new clock, we do a simple string replacement of the old clock name with a new one before executing each block.

## 1. Load extracted code blocks

In [ ]:
import pickle
import os
import sys
import subprocess
import tempfile
import re
import time
from io import StringIO
from collections import Counter

import pandas as pd
import numpy as np

In [ ]:
# ---------- Configuration ----------
# Path to the extracted code pickle
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
CODE_PKL = os.path.join(REPO_ROOT, "extracted_code.pkl")

# The code blocks reference absolute paths from the original DFS environment.
# We replace them so blocks point to the current environment.
OLD_BASE_DIR = "/dfs/user/hanchenw/Agent_PG/clockbase_agent"
NEW_BASE_DIR = REPO_ROOT  # <-- change this if your data lives elsewhere

print(f"REPO_ROOT:     {REPO_ROOT}")
print(f"CODE_PKL:      {CODE_PKL}")
print(f"OLD_BASE_DIR:  {OLD_BASE_DIR}")
print(f"NEW_BASE_DIR:  {NEW_BASE_DIR}")

In [ ]:
with open(CODE_PKL, "rb") as f:
    code_blocks = pickle.load(f)

print(f"Total studies: {len(code_blocks):,}")
print(f"Total code blocks: {sum(len(v) for v in code_blocks.values()):,}")

# Block-count distribution
counts = Counter(len(v) for v in code_blocks.values())
print(f"\nBlocks per study (min={min(counts)}, max={max(counts)}):")
for n in sorted(counts)[:15]:
    print(f"  {n:3d} blocks: {counts[n]:5d} studies")
if max(counts) > 15:
    print(f"  ... (up to {max(counts)} blocks)")

## 2. Inspect a single study's code

In [ ]:
# Use GSE203182 as demo — it has a clean, self-contained Block 0 that does the full analysis
DEMO_STUDY = "GSE203182"

demo_blocks = code_blocks[DEMO_STUDY]
print(f"Demo study: {DEMO_STUDY}  ({len(demo_blocks)} blocks)")

In [ ]:
for i, block in enumerate(demo_blocks):
    first_line = block.strip().split("\n")[0]
    print(f"\n{'='*60}")
    print(f"Block {i}  ({len(block)} chars)  | {first_line[:70]}")
    print('='*60)
    # Print first 30 lines
    lines = block.strip().split("\n")
    for line in lines[:30]:
        print(line)
    if len(lines) > 30:
        print(f"... ({len(lines) - 30} more lines)")

## 3. Define replay infrastructure

The `replay_study()` function mirrors the execution model from `src/coding_agent.py`:
- Python blocks run via `exec()` in a **persistent namespace** (lines 37-49)
- R blocks (detected by `#!R`) run via `Rscript` subprocess (lines 52-62)
- Bash blocks (detected by `#!BASH`) run via bash subprocess (lines 65-88)

In [ ]:
def _run_r_code(code: str) -> str:
    """Run R code via Rscript subprocess."""
    with tempfile.NamedTemporaryFile(suffix=".R", mode="w", delete=False) as f:
        f.write(code)
        tmp = f.name
    try:
        result = subprocess.run(
            ["Rscript", tmp], capture_output=True, text=True, check=False, timeout=300
        )
        return result.stderr if result.returncode != 0 else result.stdout
    except FileNotFoundError:
        return "Error: Rscript not found — skipping R block"
    except subprocess.TimeoutExpired:
        return "Error: R block timed out (300s)"
    finally:
        os.unlink(tmp)


def _run_bash_script(script: str) -> str:
    """Run a Bash script via subprocess."""
    with tempfile.NamedTemporaryFile(suffix=".sh", mode="w", delete=False) as f:
        if not script.startswith("#!/"):
            f.write("#!/bin/bash\nset -e\n")
        f.write(script)
        tmp = f.name
    os.chmod(tmp, 0o755)
    try:
        result = subprocess.run(
            [tmp], shell=True, capture_output=True, text=True,
            check=False, timeout=300,
        )
        if result.returncode != 0:
            return f"Error (exit {result.returncode}):\n{result.stderr}"
        return result.stdout
    except subprocess.TimeoutExpired:
        return "Error: Bash block timed out (300s)"
    finally:
        os.unlink(tmp)

In [ ]:
def replay_study(
    blocks,
    old_clock,
    new_clock,
    output_dir,
    old_base_dir=OLD_BASE_DIR,
    new_base_dir=NEW_BASE_DIR,
    timeout_per_block=300,
    verbose=True,
):
    """
    Replay a study's extracted code blocks with a different clock.

    Parameters
    ----------
    blocks : list[str]
        Code blocks extracted from the agent trace.
    old_clock : str
        Clock column name to replace (the one hardcoded in the blocks).
    new_clock : str
        Clock column name to use instead.
    output_dir : str
        Directory where results should be saved.
    old_base_dir : str
        Base directory path to replace in code blocks.
    new_base_dir : str
        Base directory path to use instead.
    timeout_per_block : int
        Max seconds per block.
    verbose : bool
        Print progress.

    Returns
    -------
    dict with keys: success (bool), outputs (list), errors (list), namespace (dict)
    """
    os.makedirs(output_dir, exist_ok=True)

    # Persistent namespace shared across all blocks (same as coding_agent.py:34-43)
    namespace = {}
    outputs = []
    errors = []
    success = True

    for i, raw_block in enumerate(blocks):
        # --- String replacements ---
        block = raw_block
        if old_clock != new_clock:
            block = block.replace(old_clock, new_clock)
        # Replace base directory (handle trailing-slash and double-slash variants)
        block = block.replace(old_base_dir + "//", new_base_dir + "/")
        block = block.replace(old_base_dir + "/", new_base_dir + "/")
        block = block.replace(old_base_dir, new_base_dir)
        # Redirect original result directories to the replay output directory.
        # Code blocks save to e.g. <base>/result_prompt3/GSE.../  or <base>/result/GSE.../ 
        # We want all output to go to output_dir instead.
        study_id = os.path.basename(output_dir)
        for result_subdir in ["result_prompt3", "result_mm_rna", "result"]:
            old_out = os.path.join(new_base_dir, result_subdir, study_id)
            if old_out in block:
                block = block.replace(old_out, output_dir)

        code = block.strip()

        # --- Detect language ---
        if code.startswith(("#!R", "# R code", "# R script")):
            lang = "R"
            code = re.sub(r"^#!R|^# R code|^# R script", "", code, 1).strip()
        elif code.startswith(("#!BASH", "# Bash script", "#!CLI")):
            lang = "Bash"
            code = re.sub(r"^#!BASH|^# Bash script|^#!CLI", "", code, 1).strip()
        else:
            lang = "Python"

        if verbose:
            first_line = code.split("\n")[0][:70]
            print(f"  Block {i}/{len(blocks)-1} [{lang}]: {first_line}")

        # --- Execute ---
        old_stdout = sys.stdout
        sys.stdout = buf = StringIO()
        error = None
        try:
            if lang == "R":
                sys.stdout = old_stdout
                out = _run_r_code(code)
                outputs.append(out)
            elif lang == "Bash":
                sys.stdout = old_stdout
                out = _run_bash_script(code)
                outputs.append(out)
            else:
                exec(code, namespace)
                out = buf.getvalue()
                outputs.append(out)
        except Exception as e:
            error = f"Block {i} error: {type(e).__name__}: {e}"
            errors.append(error)
            success = False
            if verbose:
                print(f"    ERROR: {error}", file=sys.stderr)
        finally:
            sys.stdout = old_stdout

        if error and verbose:
            print(f"    -> {error}")

    return {
        "success": success,
        "outputs": outputs,
        "errors": errors,
        "namespace": namespace,
    }

## 4. Demo: replay one study on a new clock

We swap the Rodents clock for the Mouse clock as a simple demo:
- **Old clock:** `Hazard_Rodents_All_All_scaleddiff_EN_tAge`  
- **New clock:** `Hazard_Mouse_All_All_scaleddiff_EN_tAge`

In [ ]:
OLD_CLOCK = "Hazard_Rodents_All_All_scaleddiff_EN_tAge"
NEW_CLOCK = "Hazard_Mouse_All_All_scaleddiff_EN_tAge"

# Output directory for the replayed study
REPLAY_OUTPUT = os.path.join(NEW_BASE_DIR, "replay_results", DEMO_STUDY)

print(f"Study:      {DEMO_STUDY}")
print(f"Old clock:  {OLD_CLOCK}")
print(f"New clock:  {NEW_CLOCK}")
print(f"Output dir: {REPLAY_OUTPUT}")
print(f"Blocks:     {len(demo_blocks)}")

In [ ]:
# Show what the string replacement does on the first block
sample = demo_blocks[0]
sample_replaced = sample.replace(OLD_CLOCK, NEW_CLOCK)
sample_replaced = sample_replaced.replace(OLD_BASE_DIR + "//", NEW_BASE_DIR + "/")
sample_replaced = sample_replaced.replace(OLD_BASE_DIR + "/", NEW_BASE_DIR + "/")
sample_replaced = sample_replaced.replace(OLD_BASE_DIR, NEW_BASE_DIR)

print("--- Original (first 500 chars) ---")
print(sample[:500])
print("\n--- After replacement (first 500 chars) ---")
print(sample_replaced[:500])

In [ ]:
result = replay_study(
    blocks=demo_blocks,
    old_clock=OLD_CLOCK,
    new_clock=NEW_CLOCK,
    output_dir=REPLAY_OUTPUT,
)

print(f"\nSuccess: {result['success']}")
print(f"Errors:  {len(result['errors'])}")
if result['errors']:
    for e in result['errors']:
        print(f"  - {e}")

In [ ]:
# Check for the output CSV
csv_path = os.path.join(REPLAY_OUTPUT, "aging_analysis.csv")
if os.path.isfile(csv_path):
    df = pd.read_csv(csv_path)
    print(f"aging_analysis.csv: {len(df)} rows")
    print(f"Columns: {df.columns.tolist()}")
    if "clock" in df.columns:
        print(f"Clock values: {df['clock'].unique().tolist()}")
    display(df.head(10))
else:
    print(f"No output CSV found at {csv_path}")
    # List whatever was produced
    if os.path.isdir(REPLAY_OUTPUT):
        print(f"Files in {REPLAY_OUTPUT}: {os.listdir(REPLAY_OUTPUT)}")

## 5. Batch replay

Loop over a small subset of studies to demonstrate batch replay.

In [ ]:
# Select studies whose block 0 is self-contained: has imports, clock references,
# and saves aging_analysis.csv — all within a single block.
# Many studies have fragmented blocks with cross-study variable dependencies,
# so we filter for the ones that will actually replay cleanly.
batch_studies = []
for study_id, blocks in code_blocks.items():
    b0 = blocks[0]
    if ("import pandas" in b0
            and OLD_CLOCK in b0
            and "aging_analysis.csv" in b0
            and "to_csv" in b0):
        batch_studies.append(study_id)
        if len(batch_studies) >= 10:
            break

print(f"Selected {len(batch_studies)} self-contained studies for batch replay:")
for sid in batch_studies:
    print(f"  {sid}: {len(code_blocks[sid])} blocks")

In [ ]:
batch_results = {}
t0 = time.time()

for idx, study_id in enumerate(batch_studies):
    print(f"\n[{idx+1}/{len(batch_studies)}] Replaying {study_id} "
          f"({len(code_blocks[study_id])} blocks) ...")
    out_dir = os.path.join(NEW_BASE_DIR, "replay_results", study_id)
    try:
        res = replay_study(
            blocks=code_blocks[study_id],
            old_clock=OLD_CLOCK,
            new_clock=NEW_CLOCK,
            output_dir=out_dir,
            verbose=False,
        )
        batch_results[study_id] = res
        status = "OK" if res["success"] else f"FAIL ({len(res['errors'])} errors)"
    except Exception as e:
        batch_results[study_id] = {"success": False, "errors": [str(e)], "outputs": []}
        status = f"EXCEPTION: {e}"
    print(f"    -> {status}")

elapsed = time.time() - t0
print(f"\nBatch complete in {elapsed:.1f}s")

In [ ]:
# Summary
n_success = sum(1 for r in batch_results.values() if r["success"])
n_fail = len(batch_results) - n_success

print(f"Results: {n_success} succeeded, {n_fail} failed out of {len(batch_results)} studies")
print()

summary_rows = []
for study_id, res in batch_results.items():
    csv_path = os.path.join(NEW_BASE_DIR, "replay_results", study_id, "aging_analysis.csv")
    has_csv = os.path.isfile(csv_path)
    n_rows = len(pd.read_csv(csv_path)) if has_csv else 0
    summary_rows.append({
        "study_id": study_id,
        "n_blocks": len(code_blocks[study_id]),
        "success": res["success"],
        "n_errors": len(res["errors"]),
        "has_csv": has_csv,
        "csv_rows": n_rows,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

In [ ]:
# Show errors for failed studies
for study_id, res in batch_results.items():
    if not res["success"]:
        print(f"\n--- {study_id} errors ---")
        for e in res["errors"]:
            print(f"  {e}")